# Validador de Procesos contra Arquetipos y Mapa de Capacidades

Este notebook carga un archivo de actividades de procesos, lo cruza contra el catálogo de arquetipos para identificar a cuál arquetipo pertenece cada código de proceso, y luego busca coincidencias entre las actividades y el mapa de capacidades de procesos.

## 1. Importación de librerías

Se importa `pandas`, la librería principal para manipulación de datos tabulares en este notebook.

In [48]:
import pandas as pd


## 2. Carga del archivo de actividades

Lee el archivo `data/query_actividades_proc_q_actual.csv` y lo carga en el DataFrame `df`. Este archivo contiene las actividades de procesos con sus campos: épica, período, código de proceso, procedimiento y actividad. Se imprime un resumen con el total de filas, columnas y tipos de datos.

In [49]:
# Cargar datos desde el archivo CSV
from pathlib import Path
import pandas as pd

csv_path = Path("data/query_actividades_proc_q_actual.csv")
df = pd.read_csv(csv_path)

print(f"Archivo cargado desde: {csv_path}")
print(f"DataFrame cargado con {len(df)} filas y {len(df.columns)} columnas")
print(f"\nColumnas: {list(df.columns)}")
print(f"\nPrimeras filas:")
df.head()
df.info()


Archivo cargado desde: data\query_actividades_proc_q_actual.csv
DataFrame cargado con 39033 filas y 5 columnas

Columnas: ['t1.id_epica', 't1.periodo', 'codigo_proceso', 't2.procedimiento', 't2.actividad']

Primeras filas:
<class 'pandas.DataFrame'>
RangeIndex: 39033 entries, 0 to 39032
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   t1.id_epica       39033 non-null  float64
 1   t1.periodo        39033 non-null  str    
 2   codigo_proceso    39033 non-null  str    
 3   t2.procedimiento  39033 non-null  str    
 4   t2.actividad      39011 non-null  str    
dtypes: float64(1), str(4)
memory usage: 1.5 MB


## 3. Matching con el catálogo de arquetipos

Carga el catálogo `data/arquetipos.csv` y cruza cada fila de actividades con él usando el campo `codigo_proceso`. Para cada código de proceso se identifican el arquetipo al que pertenece y su nombre. El resultado queda en `result_filtered`, que contiene únicamente las filas que tuvieron al menos una coincidencia. Las columnas `matched_*` se convierten a texto plano (sin corchetes ni comillas).

In [ ]:
# Cargar arquetipos y hacer matching con los datos de actividades
import sys, os
sys.path.insert(0, os.getcwd())

from helper.match_arquetipos import load_arquetipos, match_codes

df_arq, df_arq_expl = load_arquetipos('data/arquetipos.csv')

# `df` debe existir en el notebook (cargado desde el archivo CSV de actividades)
result, merged = match_codes(df, df_arq_expl, sql_code_col='codigo_proceso')
print(f"Filas cargadas: {len(df)}; coincidencias encontradas: {merged['proc_code'].notna().sum()}")

# Filtrar solo filas con coincidencias en arquetipos
result_filtered = result[result['matched_codigo_arquetipo'].notna()]

# Convertir listas a texto (sin corchetes ni comillas)
cols_to_fix = ['matched_proc_codes', 'matched_codigo_arquetipo', 'matched_nombre_arquetipo']
for col in cols_to_fix:
    result_filtered[col] = result_filtered[col].apply(
        lambda x: ', '.join(x) if isinstance(x, list) else x
    )

result_filtered

## 4. Exportar resultado a CSV

Guarda el DataFrame `result_filtered` en `data/output/resultado_matching_arquetipos.csv`. Si el archivo ya existe, se sobrescribe. Antes de ejecutar esta celda, asegúrate de que el archivo **no esté abierto en Excel** para evitar errores de permisos.

In [51]:
# Guardar resultados en CSV
import os

# Check if result_filtered exists and has data
if 'result_filtered' not in locals() or result_filtered is None or len(result_filtered) == 0:
    print("Advertencia: No hay resultados con coincidencias para guardar.")
else:
    output_dir = 'data/output'
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, 'resultado_matching_arquetipos.csv')
    result_filtered.to_csv(output_path, index=False, encoding='utf-8')
    print(f"Archivo guardado en: {output_path}")
    print(f"Total de filas: {len(result_filtered)}")

Archivo guardado en: data/output\resultado_matching_arquetipos.csv
Total de filas: 26667


## 5. Matching con el Mapa de Capacidades de Procesos

Compara la columna `t2.actividad` del resultado anterior contra la columna `Capacidad de Procesos (Nivel 3)` del archivo `data/mapa_capacidades.csv`. La búsqueda es por coincidencia de texto normalizado (minúsculas, sin puntuación final), en tres modalidades: exacta, actividad contenida en la capacidad, o capacidad contenida en la actividad. El resultado `df_matches` muestra cada coincidencia junto con el código de capacidad y los arquetipos relacionados del mapa.

In [52]:
# Matching: t2.actividad vs Capacidad de Procesos (Nivel 3) de mapa_capacidades.csv
import pandas as pd

df_result = pd.read_csv('data/output/resultado_matching_arquetipos.csv', dtype=str)
df_cap = pd.read_csv('data/mapa_capacidades.csv', dtype=str, encoding='latin-1')

# Normalizar texto: minúsculas, sin espacios extremos ni punto final
def normalizar(texto):
    if pd.isna(texto) or str(texto).strip() == '':
        return ''
    return str(texto).strip().lower().rstrip('.')

df_result['_act_norm'] = df_result['t2.actividad'].apply(normalizar)

# Lista de capacidades normalizadas con sus datos originales
caps = [
    (normalizar(row['Capacidad de Procesos (Nivel 3)']),
     row['Capacidad de Procesos (Nivel 3)'],
     row.get('Código', ''),
     row.get('Arquetipos Relacionado', ''))
    for _, row in df_cap.iterrows()
    if pd.notna(row['Capacidad de Procesos (Nivel 3)'])
]

matches = []
for _, row in df_result.iterrows():
    act = row['_act_norm']
    if not act:
        continue
    for cap_norm, cap_orig, cap_cod, cap_arq in caps:
        if not cap_norm:
            continue
        if act == cap_norm or act in cap_norm or cap_norm in act:
            matches.append({
                't2.actividad':                    row['t2.actividad'],
                'codigo_proceso':                  row['codigo_proceso'],
                'matched_codigo_arquetipo':        row.get('matched_codigo_arquetipo', ''),
                'matched_nombre_arquetipo':        row.get('matched_nombre_arquetipo', ''),
                'Capacidad de Procesos (Nivel 3)': cap_orig,
                'Código Capacidad':                cap_cod,
                'Arquetipos Relacionado':          cap_arq,
            })

df_matches = pd.DataFrame(matches).drop_duplicates()
print(f"Total de coincidencias encontradas: {len(df_matches)}")
df_matches

Total de coincidencias encontradas: 905


,t2.actividad,codigo_proceso,matched_codigo_arquetipo,matched_nombre_arquetipo,Capacidad de Procesos (Nivel 3),Código Capacidad,Arquetipos Relacionado
0,recibir solicitud,PAN040102,ARQ002,Abrir producto,Recibir solicitud,,NaN
2,m: gestión de servicios corporativos p: custod...,PAN040102,ARQ002,Abrir producto,Custodiar documento,,Procesar documento
4,notificar inconsistencias,PAN040102,ARQ002,Abrir producto,Notificar inconsistencias,,NaN
5,recibir solicitud de apertura,PAN040103,ARQ002,Abrir producto,Recibir solicitud,,NaN
6,notificar inconsistencias,PAN040103,ARQ002,Abrir producto,Notificar inconsistencias,,NaN
...,...,...,...,...,...,...,...
1926,generar documento [cmf04],T160669,ARQ005,Aplicar novedad,Generar documento,,Abrir producto
1927,generar documento [cmf04],T160669,ARQ005,Aplicar novedad,Generar documento,,Aplicar novedad
1930,custodiar documentos en la nube,T160669,ARQ005,Aplicar novedad,Custodiar documento,,Procesar documento
1934,custodiar documentos físicos,T160669,ARQ005,Aplicar novedad,Custodiar documento,,Procesar documento


## 6. Resumen de cobertura en el Mapa de Capacidades por código de proceso

Agrupa los resultados por `t1.id_epica`, `t1.periodo` y `codigo_proceso` para calcular cuántas actividades distintas de cada proceso tienen al menos una coincidencia en el mapa de capacidades.

- **cantidad_matches**: número de actividades únicas del proceso que aparecen en el mapa.
- **porcentaje_en_mapa**: proporción de esas actividades sobre el total de actividades del proceso.

Los procesos sin ninguna coincidencia también se incluyen con `cantidad_matches = 0` y `porcentaje_en_mapa = 0.0%`.

In [53]:
# Resumen de cobertura en el Mapa de Capacidades por código de proceso
import pandas as pd

# Total de actividades distintas por (epica, periodo, proceso)
total_por_proceso = (
    df_result
    .groupby(['t1.id_epica', 't1.periodo', 'codigo_proceso'])['t2.actividad']
    .nunique()
    .reset_index()
    .rename(columns={'t2.actividad': 'total_actividades'})
)

# Actividades distintas con match en el mapa, por proceso
matches_distintos = (
    df_matches[['codigo_proceso', 't2.actividad']]
    .drop_duplicates()
    .groupby('codigo_proceso')['t2.actividad']
    .nunique()
    .reset_index()
    .rename(columns={'t2.actividad': 'cantidad_matches'})
)

# Left join para conservar procesos sin coincidencias
resumen = total_por_proceso.merge(matches_distintos, on='codigo_proceso', how='left')
resumen['cantidad_matches'] = resumen['cantidad_matches'].fillna(0).astype(int)

# Calcular porcentaje y formatear
pct_num = (resumen['cantidad_matches'] / resumen['total_actividades'] * 100).round(1)
resumen['porcentaje_en_mapa'] = pct_num.astype(str) + '%'

# Resultado final ordenado de mayor a menor cobertura
resumen_cobertura = (
    resumen[['t1.id_epica', 't1.periodo', 'codigo_proceso', 'porcentaje_en_mapa', 'cantidad_matches']]
    .sort_values('cantidad_matches', ascending=False)
    .reset_index(drop=True)
)

print(f"Total procesos:              {len(resumen_cobertura)}")
print(f"Con coincidencias en mapa:   {(resumen_cobertura['cantidad_matches'] > 0).sum()}")
print(f"Sin coincidencias:           {(resumen_cobertura['cantidad_matches'] == 0).sum()}")
resumen_cobertura


Total procesos:              244
Con coincidencias en mapa:   198
Sin coincidencias:           46


,t1.id_epica,t1.periodo,codigo_proceso,porcentaje_en_mapa,cantidad_matches
0,6757906.0,2026Q1,T160112,5.8%,14
1,6753432.0,2026Q1,T160668,10.7%,14
2,7025580.0,2026Q1,T160668,19.7%,14
3,7014113.0,2026Q1,T160668,10.7%,14
4,7024757.0,2026Q1,T160668,10.7%,14
...,...,...,...,...,...
239,7056043.0,2026Q1,T160331,0.0%,0
240,7073179.0,2026Q1,T160148,0.0%,0
241,7072009.0,2026Q1,T160090,0.0%,0
242,7144412.0,2026Q1,T051918,0.0%,0
